# 04. GhanaTok Training (Byte-Level BPE & SentencePiece Unigram)
Trains custom, Ghanaian-language-tailored tokenizers on the combined Twi and Ewe corpus, incorporating a seed morpheme vocabulary and a dialect-aware pre-tokenizer to reduce subword fragmentation.

## 1. Vocabulary Size Decision (8,000 Tokens)

> **Design Decision & Justification**:
> We chose a vocabulary size of 8,000 for GhanaTok because our combined Twi and Ewe training corpus is in the tens of thousands of sentences, not millions. A much larger vocabulary (for example 32,000, matching XLM-R) would be mostly empty slots never seen during training, and would risk the tokenizer memorising rare training sentences rather than learning genuinely useful Twi and Ewe subword units.

## 2. Upgrades: Seed Morpheme Vocabulary & Dialect-Aware Pre-Tokenizer

### A. Seed Morphemes
Listing known Twi/Ewe affixes (`mbe-`, `n-`, `bɛ-`, `-foɔ`, `-nom`, `-yeɛ`, `-wo`, `-m`, `mega-`, `ɖe-`) and high-frequency greetings in both lowercase and capitalized forms (`Akwaaba`, `akwaaba`, `Medaase`, `medaase`, `Woezɔ`, `woezɔ`, `Aleke`, `aleke`, `Akpe`, `akpe`) guarantees them vocabulary slots.

### B. Dialect-Aware Pre-Tokenizer Regex
Twi and Ewe use extra letters beyond the standard Latin alphabet (`ɛ`, `ɔ`, `ɖ`, `ŋ`, `ʋ`, `ƒ`) plus combining tone marks. This pattern makes sure the tokenizer treats them as normal letters, not as symbols to split on, and keeps the space before a word attached to it, matching standard byte-level BPE convention.

In [6]:
import sys
sys.path.append("..")
from src.train_ghanatok import SEED_MORPHEMES, GHANA_PRETOKENIZER_PATTERN

print("Seed morphemes:", SEED_MORPHEMES)
print("Pre-tokenizer pattern:", repr(GHANA_PRETOKENIZER_PATTERN))

Seed morphemes: ['mbe-', 'n-', 'bɛ-', '-foɔ', '-nom', '-yeɛ', '-wo', '-m', 'mega-', 'ɖe-', 'akwaaba', 'medaase', 'woezɔ', 'aleke', 'akpe']
Pre-tokenizer pattern: <tokenizers.Regex object at 0x0000021A654BD4A0>


## 3. Build the Combined Training Corpus
Combines the Twi and Ewe training splits strictly from `train.txt`. Validation sets (`val.txt`) and test sets (`test.txt`) remain untouched.

In [7]:
from src.train_ghanatok import build_training_corpus

corpus_path = build_training_corpus()
with open(corpus_path, encoding="utf-8") as f:
    lines = f.readlines()

print("Total training sentences:", len(lines))
print("\nFirst 3 lines:")
for line in lines[:3]:
    print(" ", line.strip())

Combined training corpus: 7621 sentences saved to C:\Users\HP\Documents\GhanaToken\data\processed\ghanatok_train_corpus.txt
Total training sentences: 7621

First 3 lines:
  Agumadi ne apɔw-mu-teɛteɛ ho hia yiye ma nnipa akwahosan sɛnea Nhomasua ne Agumadi Dwumadibea no hyɛ ho nkuran no.
  Wɔn a wɔyi akwan no ahwɛ sɛdeɛ nsamsoɔ ka yɛn atenaeɛ
  Mmofra a wɔwɔ yɛn mpɔtam hɔ no tow wɔn ho gu wɔ petee mu sen sɛ wɔbɛkɔ tiafi mu.


## 4. Train Primary Model: GhanaTok (Byte-Level BPE)
Trains a Byte-Level BPE tokenizer using Hugging Face's low-level `Tokenizer` API with our dialect-aware pre-tokenizer sequence and seed morphemes.

In [8]:
from src.train_ghanatok import train_ghanatok

tokenizer_path = train_ghanatok()
print("BPE tokenizer ready at:", tokenizer_path)

GhanaTok (BPE) trained and saved to C:\Users\HP\Documents\GhanaToken\models\ghanatok\ghanatok_tokenizer.json
BPE tokenizer ready at: C:\Users\HP\Documents\GhanaToken\models\ghanatok\ghanatok_tokenizer.json


## 5. Taste-Test: GhanaTok (BPE)
Load the trained model back using `PreTrainedTokenizerFast` and inspect tokenization and decoding on sample Twi and Ewe sentences.

> **Note on Byte-Level Representations & Special Tokens**:
> 1. **Stand-in characters** (`sÉĽn`, `nÃ¨fÉĶ`): Byte-level BPE displays UTF-8 bytes for characters like `ɛ`, `ɔ`, `è` as printable stand-ins (like GPT-2). This is cosmetic and not text corruption.
> 2. **Isolated `Ġ` token**: Happens when a seed special token matches sentence-initially; does not affect per-word fertility evaluation.
> 3. **Decoding**: Use `skip_special_tokens=False` when reconstructing text to retain seed words.

In [9]:
from transformers import PreTrainedTokenizerFast

ghanatok = PreTrainedTokenizerFast(
    tokenizer_file=str(tokenizer_path),
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<s>",
    eos_token="</s>",
)

sample_twi = "Akwaaba, wo ho te sɛn?"
sample_ewe = "Woezɔ, aleke nèfɔ?"

print("GhanaTok (BPE) on Twi:", ghanatok.tokenize(sample_twi))
print("GhanaTok (BPE) on Ewe:", ghanatok.tokenize(sample_ewe))

encoded_ewe = ghanatok.encode(sample_ewe)
print("\nDecoded with skip_special_tokens=False:", ghanatok.decode(encoded_ewe, skip_special_tokens=False))

GhanaTok (BPE) on Twi: ['A', 'kwa', 'aba', ',', 'Ġwo', 'Ġho', 'Ġte', 'ĠsÉĽn', '?']
GhanaTok (BPE) on Ewe: ['Wo', 'e', 'zÉĶ', ',', 'Ġ', 'aleke', 'ĠnÃ¨', 'fÉĶ', '?']

Decoded with skip_special_tokens=False: Woezɔ, aleke nèfɔ?


## 6. Train GhanaTok-Unigram
Trains a SentencePiece Unigram tokenizer with `character_coverage=1.0` and `user_defined_symbols=SEED_MORPHEMES` to test probabilistic subword pruning against greedy BPE merges.

In [11]:
from src.train_ghanatok import train_ghanatok_unigram, SentencePieceWrapper

unigram_model_path = train_ghanatok_unigram()
ghanatok_unigram = SentencePieceWrapper(str(unigram_model_path))

print("\nGhanaTok-Unigram on Twi:", ghanatok_unigram.tokenize(sample_twi))
print("GhanaTok-Unigram on Ewe:", ghanatok_unigram.tokenize(sample_ewe))

GhanaTok-Unigram trained and saved to C:\Users\HP\Documents\GhanaToken\models\ghanatok_unigram\ghanatok_unigram.model

GhanaTok-Unigram on Twi: ['▁A', 'kwa', 'aba', ',', '▁wo', '▁ho', '▁te', '▁sɛn', '?']
GhanaTok-Unigram on Ewe: ['▁Wo', 'e', 'z', 'ɔ', ',', '▁', 'aleke', '▁n', 'è', 'fɔ', '?']
